# iHydroCal Workflow 02 — Run PESTPP-IES

This notebook starts from the PEST dataset created in `01_setup_pest_dataset.ipynb`.

Workflow:

1. Load config and identify `ihydrocal_workspace/main`.
2. Run initial `pestpp-glm` with the base control file.
3. Reweight observation groups.
4. Create an IES control file.
5. Run `pestpp-ies` with workers.

Recommended structure:

```text
ihydrocal_workspace/
├── main/              # clean template/model folder
├── pecos_rw_ies/      # IES master folder
├── worker_0/
├── worker_1/
└── ...
```

## 0. Imports

In [ ]:
from pathlib import Path

from ihydrocal.core.config import load_config
from ihydrocal.core.pest import (
    run_pestpp,
    reweight_pest_control_file,
    create_ies_control_file,
    run_pestpp_ies_workers,
)

## 1. Load config and paths

In [ ]:
CONFIG_FILE = Path("C:/Users/seonggpa/Documents/projects/watersheds/Pecos/Analysis/calibration/config/setup_swatplus.yml")

cfg = load_config(CONFIG_FILE)

workspace_dir = cfg["paths"]["workspace_dir"]
model_dir = workspace_dir / "main"
base_pst = cfg["pest"]["control_file"]

print(f"Workspace directory : {workspace_dir}")
print(f"Main model directory: {model_dir}")
print(f"Base PEST file      : {base_pst}")

## 2. Initial PEST++ GLM run

This checks the full forward-run chain:

```text
PEST++ → calibration.cal → forward_run.py → SWAT+ → cha_flo_out_day.csv → sim_stf_day.dat → instruction file
```

This should usually be run with `noptmax = 0` in the base control file.

In [ ]:
run_pestpp(
    model_dir=model_dir,
    pst_file=base_pst,
    pestpp_exe="pestpp-glm.exe",
)

## 3. Reweight observation groups

This balances non-zero observation groups to the selected target phi.

Output:

```text
pecos_rw.pst
```

In [ ]:
rw_pst = reweight_pest_control_file(
    model_dir=model_dir,
    pst_file=base_pst,
    output_pst_file="pecos_rw.pst",
    target_phi=1000.0,
)

print(f"Created reweighted PEST file: {rw_pst}")

## 4. Create IES control file

This creates an IES control file from the reweighted control file.

Adjust these values as needed:

- `ies_num_reals`
- `noptmax`

In [ ]:
ies_pst = create_ies_control_file(
    model_dir=model_dir,
    base_pst_file="pecos_rw.pst",
    ies_pst_file="pecos_rw_ies.pst",
    ies_num_reals=300,
    noptmax=10,
)

print(f"Created IES control file: {ies_pst}")

## 5. Run PESTPP-IES workers

`model_dir` is the clean template directory.

`master_dir` is placed under `ihydrocal_workspace`.

`worker_root=workspace_dir` creates worker folders directly under `ihydrocal_workspace`, not inside `main`.

In [ ]:
master_dir = workspace_dir / "pecos_rw_ies"

run_pestpp_ies_workers(
    model_dir=model_dir,
    ies_pst_file=ies_pst,
    master_dir=master_dir,
    worker_root=workspace_dir,
    num_workers=None,       # use physical cores if available
    pestpp_exe="pestpp-ies",
)

## 6. After the run

Check:

```text
ihydrocal_workspace/pecos_rw_ies
```

Useful files often include:

```text
*.phi.actual.csv
*.par.csv
*.obs.csv
*.rei
*.rec
```

Exact output names depend on the PEST++ version and the control-file name.